# PharmaShield GCC — Layer 1: Multi-Source Data Ingestion
Group 3 · AI in Operations. Runs plan §5.2 steps 1–3 end to end and writes `data/processed/suppliers_matrix.csv` in the locked schema (§5.1).

Run all cells top to bottom (plan step 10). GenAI disclosure: this notebook and the `layer1_ingestion` package were generated with Claude; prompts are in `docs/GENAI_PROMPT_LOG_layer1.md`.

In [1]:
import sys, json
from pathlib import Path
import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "layer1_ingestion").exists())
sys.path.insert(0, str(ROOT / "layer1_ingestion"))
import run_ingestion
from ingestion.config import SETTINGS
P = SETTINGS.paths
pd.set_option("display.max_columns", 60, "display.width", 220)
print("SKU:", SETTINGS.sku.sku_id, "| tier", SETTINGS.sku.criticality_tier, "| as of", SETTINGS.as_of)

SKU: CARBOPLATIN_INJ_10MG_ML | tier 1 | as of 2026-11-16


## Run the pipeline
Live openFDA + World Bank + FDA inspections. Step 1 prints shape and class balance at every step. Set `OFFLINE = True` to replay cached API responses.

In [2]:
# OFFLINE = True replays the cached API snapshot of 15 Nov 2026 (same results as the committed data).
# Set it to False for a live pull; then re-run Layer 3 (notebook 02), because Layer 1 rewrites the score table with C2-C4 empty.
OFFLINE = True
run_ingestion.main(["--log-level", "WARNING"] + (["--offline"] if OFFLINE else []))

[scms] STEP 1 raw file read: shape=(10324, 33) | class balance: target not built yet


2026-11-16 15:52:28,048 | WARNING | pharmashield.ingestion | [scms] exact_duplicate_lines: 4/10324 -> quarantined (kept lowest ID)


[scms] STEP 2 blank rows + duplicates removed: shape=(10320, 33) | class balance: target not built yet
[scms] STEP 3 dates parsed (errors=coerce) + target late built: shape=(10320, 40) | class balance: late=1186 (11.49%) on_time=9134
[scms] STEP 4 numeric fields coerced + impossible values removed: shape=(10320, 40) | class balance: late=1186 (11.49%) on_time=9134
[scms] STEP 5 freight/weight coerced, shared shipments allocated, insurance imputed: shape=(10320, 56) | class balance: late=1186 (11.49%) on_time=9134


2026-11-16 15:52:28,292 | WARNING | pharmashield.ingestion | [scms] date_order_violation(delivered<PO or PO<PQ): 10/10320 -> kept + flagged (exclude in ML if desired)


2026-11-16 15:52:28,297 | WARNING | pharmashield.ingestion | [scms] implausible_delay_abs_gt_365d: 4/10324 -> quarantined (likely keying error)


2026-11-16 15:52:28,456 | WARNING | pharmashield.ingestion | [openfda_ndc] duplicate_product_ndc: 7/48 -> kept latest listing


... (7 outputs left out)

2026-11-16 15:52:40,072 | WARNING | pharmashield.ingestion | [scenario] labelers_missing_lpi: 4/18 -> neutral value 3.40 (median of rankable makers with a known country) + flagged


2026-11-16 15:52:40,651 | WARNING | pharmashield.ingestion | [scenario] c1_price_outliers_capped_at_1.0: 1/18 -> min-max scaled on inliers only; raw USD kept in C1_landed_cost_usd_raw



=== Layer 1 ingestion summary ===
  scms                         OK       rows=10316   late rate 11.5%; 7771 train / 2545 test
  ghsc_psm                     SKIPPED  rows=-       optional stretch dataset (plan 3.2) not supplied
  internal_gpo                 PROXY    rows=-       confidential GPO ERP / Tatmeen data not supplied - SCMS is the proxy (plan 3)
  openfda_ndc                  OK       rows=18      18 labelers in panel (11 finished injection, 7 bulk API only)
  openfda_label                OK       rows=13      
  openfda_recalls              OK       rows=4961    
  openfda_shortages            OK       rows=35      35 SKU records (35 active); oncology current: 84 records / 8 molecules
  fda_inspections              OK       rows=9262    mode=manual_export
  worldbank                    OK       rows=217     169 countries with LPI
  warning_letters              OK       rows=50      
  contracts                    OK       rows=11      
  scenario                     OK   

0

## Step 1 — Clean SCMS training data
`late = delivered > scheduled`; dates parsed with `errors='coerce'`; non-numeric freight coerced to NaN.

In [3]:
clean = pd.read_csv(P.interim / "scms_shipments_clean.csv", low_memory=False)
print("shape:", clean.shape)
print("class balance:", clean["late"].value_counts().to_dict(), f"| late rate {clean['late'].mean():.2%}")
print(clean.groupby("time_split")["late"].agg(["size", "mean"]).round(4))
print("freight status:", clean["freight_status"].value_counts().to_dict())
clean[["id", "scheduled_delivery_date", "delivered_to_client_date", "delay_days", "late", "time_split", "shipment_mode",
       "manufacturing_country", "vendor_inco_term", "freight_cost_usd_coerced", "freight_usd", "weight_kg",
       "freight_to_value_ratio", "unit_price", "line_item_value"]].head()

shape: (10316, 74)
class balance: {0: 9130, 1: 1186} | late rate 11.50%
            size    mean
time_split              
test        2545  0.1399
train       7771  0.1068
freight status: {'numeric': 6196, 'reference': 2335, 'included_in_commodity': 1441, 'captured_separately': 239, 'ref_captured_separately': 105}


,id,scheduled_delivery_date,delivered_to_client_date,delay_days,late,time_split,shipment_mode,manufacturing_country,vendor_inco_term,freight_cost_usd_coerced,freight_usd,weight_kg,freight_to_value_ratio,unit_price,line_item_value
0,1,2006-06-02,2006-06-02,0,0,train,Air,India,EXW,780.34,780.340000,13.000000,1.416225,0.97,551.0
1,3,2006-11-14,2006-11-14,0,0,train,Air,India,EXW,4521.50,4521.500000,358.000000,0.729274,0.03,6200.0
2,4,2006-08-27,2006-08-27,0,0,train,Air,Germany,FCA,1653.78,1653.780000,171.000000,0.041344,0.80,40000.0
3,15,2006-09-01,2006-09-01,0,0,train,Air,India,EXW,16007.06,16007.060000,1855.000000,0.125683,0.07,127360.8
4,16,2006-08-11,2006-08-11,0,0,train,Air,India,EXW,45450.08,9032.818057,1508.448149,0.074283,0.05,121600.0


## Step 2 — Site & corridor features (train period only, no leakage)

In [4]:
site = pd.read_csv(P.interim / "scms_site_features_train.csv")
corridor = pd.read_csv(P.interim / "scms_corridor_features_train.csv")
display(site.sort_values("site_n_shipments", ascending=False).head(10))
display(corridor.sort_values("corridor_n_shipments", ascending=False).head(10))

,manufacturing_site,manufacturing_country,manufacturing_iso3,site_n_shipments,site_on_time_rate,site_mean_delay_days,site_p90_delay_days,site_mean_freight_to_value,site_unit_price_variance,site_unit_price_cv,site_air_share,first_shipment,last_shipment,site_low_history,country_n_shipments,country_on_time_rate,country_air_share,country_mean_freight_to_value
10,"Aurobindo Unit III, India",India,IND,2517,0.863727,-6.022249,5.4,1.180751,0.020733,1.012794,0.711164,2006-08-11,2013-12-31,False,5283,0.863714,0.656634,0.779058
62,Mylan (formerly Matrix) Nashik,India,IND,1062,0.871940,-9.548964,5.9,0.482419,0.029462,0.624847,0.652542,2008-01-08,2013-12-31,False,5283,0.863714,0.656634,0.779058
22,"Cipla, Goa, India",India,IND,533,0.849906,-5.153846,6.8,0.155524,0.019227,0.950205,0.495310,2006-09-04,2013-12-31,False,5283,0.863714,0.656634,0.779058
37,Hetero Unit III Hyderabad IN,India,IND,506,0.853755,-5.440711,8.0,0.425221,0.006944,0.733932,0.598814,2008-02-21,2013-12-31,False,5283,0.863714,0.656634,0.779058
79,"Strides, Bangalore, India.",India,IND,413,0.854722,-10.012107,5.8,0.213192,0.003617,0.673671,0.615012,2007-01-02,2013-12-31,False,5283,0.863714,0.656634,0.779058
42,Inverness Japan,Japan,JPN,344,0.933140,-0.784884,0.0,3.012242,0.112655,0.519742,0.927326,2007-08-09,2012-07-23,False,634,0.888013,0.949527,2.790201
80,"Trinity Biotech, Plc",Ireland,IRL,296,0.996622,-0.027027,0.0,0.664253,2.565756,0.902885,0.925676,2006-12-29,2013-12-23,False,296,0.996622,0.925676,0.664253
8,"Alere Medical Co., Ltd.",Japan,JPN,288,0.833333,2.069444,4.0,2.568161,1.063360,1.016650,0.979167,2011-10-05,2013-12-23,False,634,0.888013,0.949527,2.790201
6,ABBVIE Ludwigshafen Germany,Germany,DEU,241,0.937759,-1.908714,0.0,0.272019,0.039428,0.584729,0.850622,2007-02-16,2013-12-31,False,323,0.950464,0.792570,0.441633
3,ABBVIE (Abbott) Logis. UK,United Kingdom,GBR,159,0.974843,-8.635220,0.0,0.554968,0.032992,1.037743,0.704403,2006-12-05,2013-12-09,False,207,0.980676,0.652174,0.480335


,manufacturing_country,manufacturing_iso3,shipment_mode,corridor_n_shipments,corridor_on_time_rate,corridor_mean_delay_days,corridor_p90_delay_days,corridor_mean_freight_to_value,corridor_unit_price_variance,corridor_median_weight_kg,corridor_low_history
18,India,IND,Air,2972,0.882234,-4.960633,2.9,0.868715,0.023812,416.365761,False
21,India,IND,Truck,1321,0.820590,-10.945496,9.0,0.158298,0.024436,1497.438492,False
30,Japan,JPN,Air,602,0.883721,0.529900,1.0,2.837656,0.592901,794.500000,False
19,India,IND,Air Charter,497,0.869215,-17.577465,5.0,2.474716,0.018745,1722.000000,False
20,India,IND,Ocean,290,0.775862,7.510345,32.0,0.389267,0.011160,5177.500000,False
23,Ireland,IRL,Air,274,0.996350,-0.029197,0.0,0.707917,2.765401,205.000000,False
11,Germany,DEU,Air,255,0.949020,-1.772549,0.0,0.255583,0.068157,255.000000,False
22,India,IND,Unknown,203,0.985222,-0.285714,0.0,0.101314,0.044665,656.648364,False
59,United States,USA,Air,167,0.970060,-2.335329,0.0,0.299590,20.915807,221.500000,False
56,United Kingdom,GBR,Air,135,0.985185,-5.251852,0.0,0.518023,1.631187,242.000000,False


## Step 3 — 2026 oncology scoring panel (live openFDA) and entity resolution

In [5]:
labelers = pd.read_csv(P.interim / "fda_ndc_qualified_labelers.csv")
print(len(labelers), "labelers:", labelers["authorization_type"].value_counts().to_dict())
display(labelers[["labeler_name_fda", "authorization_type", "n_products", "strengths"]])
display(pd.read_csv(P.interim / "plan_vs_live_reconciliation.csv"))
display(pd.read_csv(P.interim / "entity_match_top20_manual_check.csv"))

18 labelers: {'finished_injection': 11, 'bulk_api_only': 7}


,labeler_name_fda,authorization_type,n_products,strengths
0,"Accord Healthcare, Inc.",finished_injection,1,10 mg/mL
1,Apotex Corp.,finished_injection,1,10 mg/mL
2,"AVYXA Pharma, LLC",finished_injection,2,500 mg/50mL|80 mg/8mL
3,BPI Labs LLC,finished_injection,4,10 mg/mL
4,Cipla Ltd.,bulk_api_only,1,2.2 kg/2.2kg
5,Eugia US LLC,finished_injection,4,150 mg/15mL|450 mg/45mL|50 mg/5mL|600 mg/60mL
6,"Fresenius Kabi USA, LLC",finished_injection,1,10 mg/mL
7,Gland Pharma Limited,finished_injection,4,10 mg/mL
8,Heraeus Precious Metals GmbH & Co. KG,bulk_api_only,1,1 g/g
9,Hetero Labs Limited,bulk_api_only,1,1 kg/kg


,item,plan_figure,this_run,note,status
0,SCMS shipments in training layer,"~10,300",10316,"raw file has 10,324 lines; rejected rows are i...",matches
1,Carboplatin labelers in openFDA NDC directory,18,18,"11 hold a finished-injection listing, 7 are bu...",matches
2,Carboplatin shortage records (openFDA),35,35,after keeping the latest status row per compan...,matches
3,Active oncology shortages,84,84,8 distinct molecules; the plan does not say wh...,matches
4,FDA CGMP warning letters collected,50-100,50,51 URLs shipped,matches
5,Manufacturers in suppliers_matrix.csv,18,18,one row per manufacturer x SKU,matches


,labeler_key,source,external_name,match_method,score,matched_via,relationship,needs_manual_check,manual_decision,decision,reviewer_note
0,HETERO LABS,fda_inspections,Hetero Labs Ltd. Unit VI,core_name_contained,90.0,HETERO LABS,self,True,NaN,NaN,NaN
1,HETERO LABS,fda_inspections,Hetero Drugs Limited,core_name_contained,90.0,HETERO LABS,self,True,NaN,NaN,NaN
2,HETERO LABS,fda_inspections,"Hetero Labs Limited, Vemagal",core_name_contained,90.0,HETERO LABS,self,True,NaN,NaN,NaN
3,HETERO LABS,fda_inspections,Hetero Labs Limited - Unit III,core_name_contained,90.0,HETERO LABS,self,True,NaN,NaN,NaN
4,HETERO LABS,fda_inspections,Hetero Labs Ltd. - Unit IX,core_name_contained,90.0,HETERO LABS,self,True,NaN,NaN,NaN
5,HOSPIRA,fda_inspections,Pfizer Health AB,core_name_contained,90.0,PFIZER,parent,True,NaN,NaN,NaN
6,HOSPIRA,fda_inspections,"Hospira Worldwide, LLC",core_name_contained,90.0,HOSPIRA,self,True,NaN,NaN,NaN
7,HOSPIRA,fda_inspections,Pfizer Manufacturing Deutschland GmbH,core_name_contained,90.0,PFIZER,parent,True,NaN,NaN,NaN
8,HOSPIRA,fda_inspections,Pfizer Italia S.r.l.,core_name_contained,90.0,PFIZER,parent,True,NaN,NaN,NaN
9,HOSPIRA,fda_inspections,Pfizer Ireland Pharmaceuticals Unlimited Company,core_name_contained,90.0,PFIZER,parent,True,NaN,NaN,NaN


## Locked schema check — `data/processed/suppliers_matrix.csv` (plan §5.1)

In [6]:
schema = pd.read_csv(P.reference / "locked_schema.csv")
matrix = pd.read_csv(P.processed / "suppliers_matrix.csv")
assert list(matrix.columns[:16]) == schema["column"].tolist(), "locked column names / order violated"
for c in ["C1_landed_cost", "C5_route_exposure", "hormuz_air_share"]:
    assert matrix[c].between(0, 1).all(), c
assert matrix["manufacturer_id"].is_unique and matrix["criticality_tier"].isin([1, 2, 3]).all()
matrix["shap_top3_reasons"].map(json.loads)
print("Schema OK:", matrix.shape[0], "manufacturers x 16 locked columns (+", matrix.shape[1] - 16, "supporting)")
print("Pending Layer 3:", [c for c in schema["column"] if matrix[c].isna().all()])
matrix.iloc[:, :16]

Schema OK: 18 manufacturers x 16 locked columns (+ 32 supporting)
Pending Layer 3: ['C2_delivery_risk', 'C4_compliance_risk']


,sku_id,criticality_tier,manufacturer_id,labeler_name_fda,manufacturing_country,is_probation,monthly_capacity_units,C1_landed_cost,C2_delivery_risk,C3_price_integrity,C4_compliance_risk,C5_route_exposure,hormuz_air_share,active_fda_shortage,oai_count_5yr,shap_top3_reasons
0,CARBOPLATIN_INJ_10MG_ML,1,ACCORD HEALTHCARE,"Accord Healthcare, Inc.",India,True,10000,0.0,NaN,0.0,NaN,0.272,0.85,True,1,[]
1,CARBOPLATIN_INJ_10MG_ML,1,APOTEX,Apotex Corp.,India,True,10000,0.0,NaN,0.0,NaN,0.272,0.85,True,0,[]
2,CARBOPLATIN_INJ_10MG_ML,1,AVYXA PHARMA,"AVYXA Pharma, LLC",Switzerland,True,10000,1.0,NaN,0.0,NaN,0.153,0.85,True,0,[]
3,CARBOPLATIN_INJ_10MG_ML,1,BPI LABS,BPI Labs LLC,India,True,10000,0.0,NaN,0.0,NaN,0.272,0.85,False,0,[]
4,CARBOPLATIN_INJ_10MG_ML,1,CIPLA,Cipla Ltd.,India,False,10000,0.0,NaN,NaN,NaN,0.272,0.85,False,2,[]
5,CARBOPLATIN_INJ_10MG_ML,1,EUGIA US,Eugia US LLC,India,True,10000,0.0,NaN,0.0,NaN,0.272,0.85,True,0,[]
6,CARBOPLATIN_INJ_10MG_ML,1,FRESENIUS KABI USA,"Fresenius Kabi USA, LLC",Unknown,True,10000,0.0,NaN,0.0,NaN,0.272,0.85,True,0,[]
7,CARBOPLATIN_INJ_10MG_ML,1,GLAND PHARMA,Gland Pharma Limited,India,True,10000,0.0,NaN,0.0,NaN,0.272,0.85,True,0,[]
8,CARBOPLATIN_INJ_10MG_ML,1,HERAEUS PRECIOUS METALS AND,Heraeus Precious Metals GmbH & Co. KG,Germany,True,10000,0.0,NaN,0.0,NaN,0.153,0.85,False,0,[]
9,CARBOPLATIN_INJ_10MG_ML,1,HETERO LABS,Hetero Labs Limited,India,False,10000,0.0,NaN,NaN,NaN,0.272,0.85,False,0,[]


## Inputs handed to the ML / NLP engines

In [7]:
display(pd.read_csv(P.interim / "panel_inference_features.csv"))      # C2 feature-based inference (plan 3.1)
display(pd.read_csv(P.processed / "quotes_2026_carboplatin.csv"))     # C1 (plan 4.3)
for name in ["warning_letters_corpus", "contracts_corpus", "fda_recalls_5yr", "fda_inspections_drugs_5yr",
             "panel_facility_inspections", "fda_shortages_sku", "wb_country_indicators"]:
    f = P.interim / f"{name}.csv"
    print(f"{name:32s}", len(pd.read_csv(f, low_memory=False)) if f.stat().st_size > 1 else 0, "rows")

,sku_id,manufacturer_id,labeler_name_fda,manufacturing_country,manufacturing_iso3,shipment_mode,is_air,air_share,freight_to_value_ratio,line_item_quantity,line_item_value,unit_price,weight_kg,vendor_inco_term,product_group,site_prior_n_shipments,site_prior_on_time_rate,site_prior_mean_delay_days,site_prior_p90_delay_days,site_prior_available,country_n_shipments,country_on_time_rate,country_air_share,country_mean_freight_to_value,corridor_n_shipments,corridor_on_time_rate,corridor_p90_delay_days,scenario_fields
0,CARBOPLATIN_INJ_10MG_ML,ACCORD HEALTHCARE,"Accord Healthcare, Inc.",India,IND,Air,True,0.55,0.0,10000,279100.0,27.91,1200.0,DDP,ONCOLOGY_INJ,0,NaN,NaN,NaN,False,5283,0.863714,0.656634,0.779058,2972,0.882234,2.9,shipment_mode|air_share|freight_to_value_ratio...
1,CARBOPLATIN_INJ_10MG_ML,APOTEX,Apotex Corp.,India,IND,Air,True,0.55,0.0,10000,279100.0,27.91,1200.0,DDP,ONCOLOGY_INJ,0,NaN,NaN,NaN,False,5283,0.863714,0.656634,0.779058,2972,0.882234,2.9,shipment_mode|air_share|freight_to_value_ratio...
2,CARBOPLATIN_INJ_10MG_ML,AVYXA PHARMA,"AVYXA Pharma, LLC",Switzerland,CHE,Air,True,0.85,0.0,10000,23517000.0,2351.70,1200.0,DDP,ONCOLOGY_INJ,0,NaN,NaN,NaN,False,48,1.000000,0.729167,0.418342,35,1.000000,0.0,shipment_mode|air_share|freight_to_value_ratio...
3,CARBOPLATIN_INJ_10MG_ML,BPI LABS,BPI Labs LLC,India,IND,Air,True,0.55,0.0,10000,279100.0,27.91,1200.0,DDP,ONCOLOGY_INJ,0,NaN,NaN,NaN,False,5283,0.863714,0.656634,0.779058,2972,0.882234,2.9,shipment_mode|air_share|freight_to_value_ratio...
4,CARBOPLATIN_INJ_10MG_ML,CIPLA,Cipla Ltd.,India,IND,Air,True,0.55,0.0,10000,279100.0,27.91,1200.0,DDP,ONCOLOGY_INJ,575,0.855652,-5.086957,5.6,True,5283,0.863714,0.656634,0.779058,2972,0.882234,2.9,shipment_mode|air_share|freight_to_value_ratio...
5,CARBOPLATIN_INJ_10MG_ML,EUGIA US,Eugia US LLC,India,IND,Air,True,0.55,0.0,10000,279100.0,27.91,1200.0,DDP,ONCOLOGY_INJ,0,NaN,NaN,NaN,False,5283,0.863714,0.656634,0.779058,2972,0.882234,2.9,shipment_mode|air_share|freight_to_value_ratio...
6,CARBOPLATIN_INJ_10MG_ML,FRESENIUS KABI USA,"Fresenius Kabi USA, LLC",Unknown,NaN,Air,True,0.55,0.0,10000,279100.0,27.91,1200.0,DDP,ONCOLOGY_INJ,0,NaN,NaN,NaN,False,17,1.000000,0.823529,0.127920,14,1.000000,0.0,shipment_mode|air_share|freight_to_value_ratio...
7,CARBOPLATIN_INJ_10MG_ML,GLAND PHARMA,Gland Pharma Limited,India,IND,Air,True,0.55,0.0,10000,279100.0,27.91,1200.0,DDP,ONCOLOGY_INJ,1,1.000000,0.000000,0.0,False,5283,0.863714,0.656634,0.779058,2972,0.882234,2.9,shipment_mode|air_share|freight_to_value_ratio...
8,CARBOPLATIN_INJ_10MG_ML,HERAEUS PRECIOUS METALS AND,Heraeus Precious Metals GmbH & Co. KG,Germany,DEU,Air,True,0.80,0.0,10000,279100.0,27.91,1200.0,DDP,ONCOLOGY_INJ,0,NaN,NaN,NaN,False,323,0.950464,0.792570,0.441633,255,0.949020,0.0,shipment_mode|air_share|freight_to_value_ratio...
9,CARBOPLATIN_INJ_10MG_ML,HETERO LABS,Hetero Labs Limited,India,IND,Air,True,0.55,0.0,10000,279100.0,27.91,1200.0,DDP,ONCOLOGY_INJ,521,0.854127,-5.406910,8.0,True,5283,0.863714,0.656634,0.779058,2972,0.882234,2.9,shipment_mode|air_share|freight_to_value_ratio...


,labeler_key,landed_cost_usd_per_vial,sku_id,labeler_name_fda,reference_vial,base_unit_price_usd,price_benchmark_id,manufacturing_country,freight_usd_per_vial,insurance_usd_per_vial,insurance_rate,insurance_basis,corridor_row_used,is_scenario_assumption
0,ACCORD HEALTHCARE,27.9546,CARBOPLATIN_INJ_10MG_ML,"Accord Healthcare, Inc.",450 mg / 45 mL,27.91,CMS_ASP_J9045_2026Q4,India,0.0,0.0446,0.0016,"median insurance/value ratio, cleaned SCMS lin...",IND,True
1,APOTEX,27.9546,CARBOPLATIN_INJ_10MG_ML,Apotex Corp.,450 mg / 45 mL,27.91,CMS_ASP_J9045_2026Q4,India,0.0,0.0446,0.0016,"median insurance/value ratio, cleaned SCMS lin...",IND,True
2,AVYXA PHARMA,2355.4623,CARBOPLATIN_INJ_10MG_ML,"AVYXA Pharma, LLC",450 mg / 45 mL,2351.70,CMS_ASP_J9278_2026Q4,Switzerland,0.0,3.7623,0.0016,"median insurance/value ratio, cleaned SCMS lin...",CHE,True
3,BPI LABS,27.9546,CARBOPLATIN_INJ_10MG_ML,BPI Labs LLC,450 mg / 45 mL,27.91,CMS_ASP_J9045_2026Q4,India,0.0,0.0446,0.0016,"median insurance/value ratio, cleaned SCMS lin...",IND,True
4,CIPLA,27.9546,CARBOPLATIN_INJ_10MG_ML,Cipla Ltd.,450 mg / 45 mL,27.91,CMS_ASP_J9045_2026Q4,India,0.0,0.0446,0.0016,"median insurance/value ratio, cleaned SCMS lin...",IND,True
5,EUGIA US,27.9546,CARBOPLATIN_INJ_10MG_ML,Eugia US LLC,450 mg / 45 mL,27.91,CMS_ASP_J9045_2026Q4,India,0.0,0.0446,0.0016,"median insurance/value ratio, cleaned SCMS lin...",IND,True
6,FRESENIUS KABI USA,27.9546,CARBOPLATIN_INJ_10MG_ML,"Fresenius Kabi USA, LLC",450 mg / 45 mL,27.91,CMS_ASP_J9045_2026Q4,Unknown,0.0,0.0446,0.0016,"median insurance/value ratio, cleaned SCMS lin...",median_of_known_rankable_makers,True
7,GLAND PHARMA,27.9546,CARBOPLATIN_INJ_10MG_ML,Gland Pharma Limited,450 mg / 45 mL,27.91,CMS_ASP_J9045_2026Q4,India,0.0,0.0446,0.0016,"median insurance/value ratio, cleaned SCMS lin...",IND,True
8,HERAEUS PRECIOUS METALS AND,27.9546,CARBOPLATIN_INJ_10MG_ML,Heraeus Precious Metals GmbH & Co. KG,450 mg / 45 mL,27.91,CMS_ASP_J9045_2026Q4,Germany,0.0,0.0446,0.0016,"median insurance/value ratio, cleaned SCMS lin...",DEU,True
9,HETERO LABS,27.9546,CARBOPLATIN_INJ_10MG_ML,Hetero Labs Limited,450 mg / 45 mL,27.91,CMS_ASP_J9045_2026Q4,India,0.0,0.0446,0.0016,"median insurance/value ratio, cleaned SCMS lin...",IND,True


warning_letters_corpus           50 rows
contracts_corpus                 11 rows
fda_recalls_5yr                  4961 rows
fda_inspections_drugs_5yr        9262 rows
panel_facility_inspections       33 rows
fda_shortages_sku                35 rows
wb_country_indicators            217 rows


## Data-quality report

In [8]:
rep = json.loads((P.quality / "quality_report.json").read_text())
display(pd.DataFrame([{"source": k, **v} for k, v in rep["sources"].items()]))
chk = pd.DataFrame(rep["checks"])
chk[chk["severity"].isin(["error", "warning"])][["source", "check", "n_affected", "n_total", "severity", "action", "detail"]]

,source,status,rows,note
0,scms,OK,10316.0,late rate 11.5%; 7771 train / 2545 test
1,ghsc_psm,SKIPPED,NaN,optional stretch dataset (plan 3.2) not supplied
2,internal_gpo,PROXY,NaN,confidential GPO ERP / Tatmeen data not suppli...
3,openfda_ndc,OK,18.0,"18 labelers in panel (11 finished injection, 7..."
4,openfda_label,OK,13.0,NaN
5,openfda_recalls,OK,4961.0,NaN
6,openfda_shortages,OK,35.0,35 SKU records (35 active); oncology current: ...
7,fda_inspections,OK,9262.0,mode=manual_export
8,worldbank,OK,217.0,169 countries with LPI
9,warning_letters,OK,50.0,NaN


,source,check,n_affected,n_total,severity,action,detail
5,scms,exact_duplicate_lines,4,10324.0,warning,quarantined (kept lowest ID),
40,scms,date_order_violation(delivered<PO or PO<PQ),10,10320.0,warning,kept + flagged (exclude in ML if desired),
42,scms,implausible_delay_abs_gt_365d,4,10324.0,warning,quarantined (likely keying error),
55,openfda_ndc,duplicate_product_ndc,7,48.0,warning,kept latest listing,
69,openfda_label,labels_without_country,10,13.0,warning,manual override via reference/manufacturer_ove...,
71,openfda_recalls,missing_key_fields,1,4962.0,warning,quarantined,
73,openfda_recalls,unparseable_classification,1,4961.0,warning,,
93,worldbank,wgi_political_stability_out_of_range,24,3904.0,warning,quarantined,
109,scenario,labelers_without_plant_country,4,18.0,warning,NEUTRAL values used (median of rankable makers...,"Fresenius Kabi USA, LLC, Sicor de Mexico S.A. ..."
111,scenario,labelers_missing_lpi,4,18.0,warning,neutral value 3.40 (median of rankable makers ...,
